## Задача 1

Лесник решил провести кластеризацию животных по их расположению в лесу. Он разделил карту на квадраты по километровым отметкам: первый квадрат можно описать 0 ≤ x ≤ 1, 0 ≤ y ≤ 1, второй — 1 ≤ x ≤ 2,0 ≤ y ≤ 1 и так далее.

Для файла А леснику нужно определить два соседних квадрата, в которых суммарно находится больше всего животных. Для файла Б леснику нужно определить три соседних квадрата. Квадраты называются соседними, если у них есть общая граница.

Для каждого файла вычислите два числа: S — количество социальных животных в выбранных соседних квадратах, и K — количество остальных животных в выбранных соседних квадратах. Животное называется социальным, если в радиусе 0.1 вокруг него находится как минимум 14 других животных.

В ответе запишите четыре числа: в первой строке S и K для файла А, во второй строке аналогичные данные для файла Б.

In [9]:
import math

def read(file):
    pts = []
    with open(file, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                pts.append((float(parts[0]), float(parts[1])))
    return pts

def is_social(pts, idx, rad=0.1, need=14):
    x, y = pts[idx]
    cnt = 0
    for i, (px, py) in enumerate(pts):
        if i == idx:
            continue
        if math.hypot(x - px, y - py) <= rad:
            cnt += 1
            if cnt >= need:
                return True
    return False

def solve_task1(fa, fb):
    pts_a = read(fa)
    n_a = len(pts_a)
    soc_a = [is_social(pts_a, i) for i in range(n_a)]
    
    best_s = -1
    best_k = -1
    
    for i in range(0, 100):
        for j in range(i + 1, 100):
            if abs(i // 10 - j // 10) + abs(i % 10 - j % 10) == 1:
                s = 0
                k = 0
                for idx, (x, y) in enumerate(pts_a):
                    cx = int(x)
                    cy = int(y)
                    cell = cy * 10 + cx
                    if cell == i or cell == j:
                        if soc_a[idx]:
                            s += 1
                        else:
                            k += 1
                if s + k > best_s + best_k:
                    best_s = s
                    best_k = k
    
    pts_b = read(fb)
    n_b = len(pts_b)
    soc_b = [is_social(pts_b, i) for i in range(n_b)]
    
    best_sb = -1
    best_kb = -1
    
    for i in range(0, 100):
        for j in range(i + 1, 100):
            for k in range(j + 1, 100):
                cells = [i, j, k]
                found = False
                for a in cells:
                    for b in cells:
                        if a != b and abs(a // 10 - b // 10) + abs(a % 10 - b % 10) == 1:
                            found = True
                            break
                    if found:
                        break
                if found:
                    s = 0
                    k_cnt = 0
                    for idx, (x, y) in enumerate(pts_b):
                        cx = int(x)
                        cy = int(y)
                        cell = cy * 10 + cx
                        if cell in cells:
                            if soc_b[idx]:
                                s += 1
                            else:
                                k_cnt += 1
                    if s + k_cnt > best_sb + best_kb:
                        best_sb = s
                        best_kb = k_cnt
    
    return best_s, best_k, best_sb, best_kb

## Задача 2

Научно﻿-﻿исследовательский институт проводит мониторинг экологического состояния различных регионов. Результаты измерений представляются в виде пары чисел: первое — концентрация загрязняющего вещества в почве, второе — концентрация того же вещества в близлежащем водоёме. Для анализа результатов эта пара рассматривается как координаты точки на плоскости, и строится график с точками, которые соответствуют всем измерениям.

По ошибке данные нескольких исследуемых регионов были записаны в один файл. Известно, что измерения относятся к одному региону, если они образуют компактные группы точек на графике. Каждая группа лежит внутри прямоугольника высотой H и шириной.

Перед проведением основного анализа необходимо очистить данные от случайных выбросов при измерении. Для этого используется метод межквартильного размаха:
* для каждой группы точек вычисляется первый квартиль Q1 (значение, ниже которого находится 25% измерений) и третий квартиль Q3 (значение, выше которого находится 25% измерений) отдельно для рядов значений концентрации загрязняющего вещества в почве и в близлежащем водоёме (координаты X и Y)
* вычисляется межквартильный размах IQR - Q3 - Q1 для каждой кординаты
* точка считается выбросом, если хотя бы одна из её координат Х или Y выходит за пределы диапазона |Q1 - 1.5 * IQR; Q3 + 1.5 * IQR|

Для каждого региона необходимо рассчитать индекс экологической опасности I, который определяется как отношение среднего значения измерений к их количеству после удаления выбросов.

Под средним значением измерений в этом случае понимается среднее евклидово расстояние между всеми парами различных точек (измерений) в регионе.

В файле A хранятся данные о трёх регионах, где H=30, W=35. Каждая строка файла содержит два числа: координаты X (концентрация в почве) и Y (концентрация в водоёме), соответствующие одному измерению. Значения даны в условных единицах. Известно, что количество измерений не превышает 1000.

В файле B хранятся данные о пяти регионах, где H=40, W=32. Известно, что количество измерений не превышает 10000. Структура хранения информации в файле B аналогична файлу А.

Для каждого файла определите:
* общее количество выявленных выбросов
* регион с максимальным индексом экологической опасности

В ответе запишите четыре числа: в первой строке — общее количество выбросов и целую часть произведения I×100000 для файла A, во второй строке — аналогичные данные для файла B.

Расчёт квартилей
* Найдите медиану значений данных. Это второй квартиль Q2
* Найдите медиану значений данных, которые находятся ниже второго квартиля. Это первый квартиль Q1
* Найдите медиану значений данных, которые выше второго квартиля. Это третий квартиль Q3

In [5]:
import numpy as np
import math

def read(fname):
    pts = []
    with open(fname, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                pts.append((float(parts[0]), float(parts[1])))
    return pts

def find_clust(pts, h, w):
    clusts = []
    used = [False] * len(pts)
    
    for i in range(len(pts)):
        if used[i]:
            continue
        clusts.append([i])
        used[i] = True
        q = [i]
        
        while q:
            cur = q.pop(0)
            cx, cy = pts[cur]
            for j in range(len(pts)):
                if not used[j] and abs(cx - pts[j][0]) <= w and abs(cy - pts[j][1]) <= h:
                    clusts[-1].append(j)
                    used[j] = True
                    q.append(j)
    return clusts

def quart(vals):
    s = sorted(vals)
    q2 = np.median(s)
    low = [x for x in s if x < q2]
    q1 = np.median(low) if low else s[0]
    high = [x for x in s if x > q2]
    q3 = np.median(high) if high else s[-1]
    return q1, q3

def filter_pts(pts):
    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]
    
    q1x, q3x = quart(xs)
    q1y, q3y = quart(ys)
    
    iqr_x = q3x - q1x
    iqr_y = q3y - q1y
    
    lo_x = q1x - 1.5 * iqr_x
    hi_x = q3x + 1.5 * iqr_x
    lo_y = q1y - 1.5 * iqr_y
    hi_y = q3y + 1.5 * iqr_y
    
    return [(x, y) for x, y in pts if lo_x <= x <= hi_x and lo_y <= y <= hi_y]

def avg_dist(pts):
    if len(pts) < 2:
        return 0
    total = 0
    cnt = 0
    for i in range(len(pts)):
        for j in range(i + 1, len(pts)):
            total += math.hypot(pts[i][0] - pts[j][0], pts[i][1] - pts[j][1])
            cnt += 1
    return total / cnt if cnt else 0

def solve_task2(file_a, file_b):
    pts_a = read(file_a)
    clusts_a = find_clust(pts_a, 30, 35)
    
    out_a = 0
    best_a = -1
    
    for cl in clusts_a:
        cl_pts = [pts_a[i] for i in cl]
        filt = filter_pts(cl_pts)
        out_a += len(cl_pts) - len(filt)
        
        if len(filt) >= 2:
            score = avg_dist(filt) / len(filt)
            if score > best_a:
                best_a = score
    
    pts_b = read(file_b)
    clusts_b = find_clust(pts_b, 40, 32)
    
    out_b = 0
    best_b = -1
    
    for cl in clusts_b:
        cl_pts = [pts_b[i] for i in cl]
        filt = filter_pts(cl_pts)
        out_b += len(cl_pts) - len(filt)
        
        if len(filt) >= 2:
            score = avg_dist(filt) / len(filt)
            if score > best_b:
                best_b = score
    
    return out_a, int(best_a * 100000), out_b, int(best_b * 100000)

## Задача 3

Кластеризуйте как в предыдущих домашках. Будем называть центром кластера точку этого кластера, сумма расстояний от которой до всех остальных точек кластера минимальна.

Для каждого файла определите координаты центра каждого кластера, затем вычислите два числа: A - среднее арифметическое абсцисс центров кластеров, и среднее B - арифметическое ординат центров кластеров. В ответе запишите четыре числа: в первой строке сначала целую часть произведения A\*10000 затем целую часть произведения B\*10000 для файла А, во второй строке — аналогичные данные для файла Б.


In [6]:
import math

def read(file):
    pts = []
    with open(file, 'r') as f:
        for line in f:
            line = line.strip().replace(',', '.')
            parts = line.split()
            if len(parts) >= 2:
                pts.append((float(parts[0]), float(parts[1])))
    return pts

def find(pts):
    groups = []
    seen = [False] * len(pts)
    
    for i in range(len(pts)):
        if seen[i]:
            continue
        groups.append([i])
        seen[i] = True
        queue = [i]
        
        while queue:
            curr = queue.pop(0)
            cx, cy = pts[curr]
            for j in range(len(pts)):
                if not seen[j]:
                    px, py = pts[j]
                    if math.hypot(cx - px, cy - py) <= 1.0:
                        groups[-1].append(j)
                        seen[j] = True
                        queue.append(j)
    return groups

def center(idx_list, all_pts):
    best = idx_list[0]
    best_sum = float('inf')
    
    for i in idx_list:
        total = 0
        xi, yi = all_pts[i]
        for j in idx_list:
            if i == j:
                continue
            xj, yj = all_pts[j]
            total += math.hypot(xi - xj, yi - yj)
        if total < best_sum:
            best_sum = total
            best = i
    
    return all_pts[best]

def solve_task3(file_a, file_b):
    pts_a = read(file_a)
    groups_a = find(pts_a)
    
    centers_a = []
    for g in groups_a:
        if g:
            centers_a.append(center(g, pts_a))
    
    avg_x_a = sum(c[0] for c in centers_a) / len(centers_a)
    avg_y_a = sum(c[1] for c in centers_a) / len(centers_a)
    
    res_ax = int(avg_x_a * 10000)
    res_ay = int(avg_y_a * 10000)
    
    pts_b = read(file_b)
    groups_b = find(pts_b)
    
    centers_b = []
    for g in groups_b:
        if g:
            centers_b.append(center(g, pts_b))
    
    avg_x_b = sum(c[0] for c in centers_b) / len(centers_b)
    avg_y_b = sum(c[1] for c in centers_b) / len(centers_b)
    
    res_bx = int(avg_x_b * 10000)
    res_by = int(avg_y_b * 10000)
    
    return res_ax, res_ay, res_bx, res_by

## Задача 4

Исследователь анализирует набор объектов, каждый из которых характеризуется пятью числовыми параметрами. Он знает, что объекты образуют несколько групп (кластеров), которые можно выявить при проекции на плоскость только двух параметров из пяти. Значения в одном из столбцов будут соответствовать координатам по оси абсцисс, а из второго — координатам по оси ординат. Каждый кластер можно заключить в квадратную область заданного размера L, причём эти квадраты между собой не пересекаются. Стороны квадратов параллельны координатным осям. Каждый объект должен принадлежать только одному кластеру.

Евклидово расстояние

В файле A хранятся данные о наборе объектов, образующих три кластера. В каждой строке через пробел записаны пять параметров, характеризующих один объект. Все значения представлены с точностью до двух знаков после запятой. Количество объектов в файле А не превышает 1000.

В файле Б записаны данные о наборе объектов, образующих шесть кластеров, с аналогичной структурой хранения информации. Количество объектов в файле Б не превышает 10000.

Для каждого файла необходимо определить, какая пара параметров позволяет разделить объекты на кластеры, и найти минимальный размер стороны квадрата L, который может содержать все точки одного кластера при проекции на плоскость найденных параметров. Также определите в каждом кластере расстояние между двумя объектами, расположенными дальше всего друг от друга, и вычислите P — среднее арифметическое таких расстояний для всех кластеров.

В ответе запишите четыре числа: в первой строке — целую часть произведения L×10000, затем целую часть произведения P×10000 для файла А, во второй строке — аналогичные значения для файла Б.

In [12]:
def solve_task4(fa, fb):
    def read(f):
        d = []
        with open(f) as i:
            for l in i:
                p = l.strip().split()
                if len(p) >= 5:
                    d.append([float(x) for x in p[:5]])
        return d

    def clust(pts, d1, d2, L):
        n = len(pts)
        used = [False]*n
        groups = []
        for i in range(n):
            if not used[i]:
                cur = [i]
                used[i] = True
                q = [i]
                while q:
                    cur_i = q.pop(0)
                    cx, cy = pts[cur_i][d1], pts[cur_i][d2]
                    for j in range(n):
                        if not used[j] and abs(pts[j][d1] - cx) <= L and abs(pts[j][d2] - cy) <= L:
                            used[j] = True
                            cur.append(j)
                            q.append(j)
                groups.append(cur)
        return groups

    def diam(pts, d1, d2):
        mx = 0
        for i in range(len(pts)):
            for j in range(i+1, len(pts)):
                d = ((pts[i][d1]-pts[j][d1])**2 + (pts[i][d2]-pts[j][d2])**2)**0.5
                if d > mx:
                    mx = d
        return mx

    def proc(pts, need):
        best_L = 100000
        best_d = (0,1)
        final_groups = []
        for d1 in range(5):
            for d2 in range(d1+1, 5):
                for L in range(1, 200):
                    Lf = L / 100.0
                    g = clust(pts, d1, d2, Lf)
                    if len(g) == need:
                        if Lf < best_L:
                            best_L = Lf
                            best_d = (d1, d2)
                            final_groups = g
                        break
        diams_list = []
        for g in final_groups:
            g_pts = [pts[i] for i in g]
            diams_list.append(diam(g_pts, best_d[0], best_d[1]))
        P = sum(diams_list) / len(diams_list)
        return int(best_L * 10000), int(P * 10000)

    res_aL, res_aP = proc(read(fa), 3)
    res_bL, res_bP = proc(read(fb), 6)
    return res_aL, res_aP, res_bL, res_bP

In [18]:
import math

def read(file):
    pts = []
    with open(file, 'r') as f:
        for line in f:
            line = line.strip().replace(',', '.')
            parts = line.split()
            if len(parts) >= 2:
                pts.append((float(parts[0]), float(parts[1])))
    return pts

def find(pts):
    groups = []
    seen = [False] * len(pts)
    
    for i in range(len(pts)):
        if seen[i]:
            continue
        group = [i]
        seen[i] = True
        queue = [i]
        
        while queue:
            cur = queue.pop(0)
            cx, cy = pts[cur]
            for j in range(len(pts)):
                if not seen[j]:
                    px, py = pts[j]
                    if math.hypot(cx - px, cy - py) <= 0.5:
                        group.append(j)
                        seen[j] = True
                        queue.append(j)
        groups.append(group)
    
    return groups

def center(idx_list, all_pts):
    best_i = idx_list[0]
    best_sum = float('inf')
    
    for i in idx_list:
        total = 0
        xi, yi = all_pts[i]
        for j in idx_list:
            if i == j:
                continue
            xj, yj = all_pts[j]
            total += math.hypot(xi - xj, yi - yj)
        if total < best_sum:
            best_sum = total
            best_i = i
    
    return all_pts[best_i]

def solve_task5(fa, fb):
    pts_a = read(fa)
    groups_a = find(pts_a)
    
    sum_x_a = 0
    sum_y_a = 0
    for g in groups_a:
        if g:
            cx, cy = center(g, pts_a)
            sum_x_a += cx
            sum_y_a += cy
    
    avg_x_a = sum_x_a / len(groups_a)
    avg_y_a = sum_y_a / len(groups_a)
    
    rax = int(avg_x_a * 10000)
    ray = int(avg_y_a * 10000)
    
    pts_b = read(fb)
    groups_b = find(pts_b)
    
    sum_x_b = 0
    sum_y_b = 0
    for g in groups_b:
        if g:
            cx, cy = center(g, pts_b)
            sum_x_b += cx
            sum_y_b += cy
    
    avg_x_b = sum_x_b / len(groups_b)
    avg_y_b = sum_y_b / len(groups_b)
    
    rbx = int(avg_x_b * 10000)
    rby = int(avg_y_b * 10000)
    
    return rax, ray, rbx, rby

In [20]:
import math

def read(file):
    pts = []
    with open(file, 'r') as f:
        for line in f:
            line = line.strip().replace(',', '.')
            parts = line.split()
            if len(parts) >= 2:
                pts.append((float(parts[0]), float(parts[1])))
    return pts

def find(pts):
    groups = []
    seen = [False] * len(pts)
    
    for i in range(len(pts)):
        if seen[i]:
            continue
        group = [i]
        seen[i] = True
        queue = [i]
        
        while queue:
            cur = queue.pop(0)
            cx, cy = pts[cur]
            for j in range(len(pts)):
                if not seen[j]:
                    px, py = pts[j]
                    if math.hypot(cx - px, cy - py) <= 0.5:
                        group.append(j)
                        seen[j] = True
                        queue.append(j)
        groups.append(group)
    
    return groups

def center(idx_list, all_pts):
    best_i = idx_list[0]
    best_sum = float('inf')
    
    for i in idx_list:
        total = 0
        xi, yi = all_pts[i]
        for j in idx_list:
            if i == j:
                continue
            xj, yj = all_pts[j]
            total += math.hypot(xi - xj, yi - yj)
        if total < best_sum:
            best_sum = total
            best_i = i
    
    return all_pts[best_i]

def solve_task5(fa, fb):
    pts_a = read(fa)
    groups_a = find(pts_a)
    
    sum_x_a = 0
    sum_y_a = 0
    for g in groups_a:
        if g:
            cx, cy = center(g, pts_a)
            sum_x_a += cx
            sum_y_a += cy
    
    avg_x_a = sum_x_a / len(groups_a)
    avg_y_a = sum_y_a / len(groups_a)
    
    rax = int(avg_x_a * 10000)
    ray = int(avg_y_a * 10000)
    
    pts_b = read(fb)
    groups_b = find(pts_b)
    
    sum_x_b = 0
    sum_y_b = 0
    for g in groups_b:
        if g:
            cx, cy = center(g, pts_b)
            sum_x_b += cx
            sum_y_b += cy
    
    avg_x_b = sum_x_b / len(groups_b)
    avg_y_b = sum_y_b / len(groups_b)
    
    rbx = int(avg_x_b * 10000)
    rby = int(avg_y_b * 10000)
    
    return rax, ray, rbx, rby

def solve_task4(fa, fb):
    def read4(f):
        d = []
        with open(f) as i:
            for l in i:
                p = l.strip().split()
                if len(p) >= 5:
                    d.append([float(x) for x in p[:5]])
        return d

    def clust(pts, d1, d2, L):
        n = len(pts)
        used = [False]*n
        groups = []
        for i in range(n):
            if not used[i]:
                cur = [i]
                used[i] = True
                q = [i]
                while q:
                    cur_i = q.pop(0)
                    cx, cy = pts[cur_i][d1], pts[cur_i][d2]
                    for j in range(n):
                        if not used[j] and abs(pts[j][d1] - cx) <= L and abs(pts[j][d2] - cy) <= L:
                            used[j] = True
                            cur.append(j)
                            q.append(j)
                groups.append(cur)
        return groups

    def diam(pts, d1, d2):
        mx = 0
        for i in range(len(pts)):
            for j in range(i+1, len(pts)):
                d = math.hypot(pts[i][d1]-pts[j][d1], pts[i][d2]-pts[j][d2])
                if d > mx:
                    mx = d
        return mx

    def proc(pts, need):
        best_L = 100000
        best_d = (0,1)
        final_groups = []
        for d1 in range(5):
            for d2 in range(d1+1, 5):
                for L in range(1, 500):
                    Lf = L / 100.0
                    g = clust(pts, d1, d2, Lf)
                    if len(g) == need:
                        if Lf < best_L:
                            best_L = Lf
                            best_d = (d1, d2)
                            final_groups = g
                        break
        if len(final_groups) == 0 or best_L == 100000:
            return 0, 0
        diams_list = []
        for g in final_groups:
            if len(g) > 1:
                g_pts = [pts[i] for i in g]
                diams_list.append(diam(g_pts, best_d[0], best_d[1]))
            else:
                diams_list.append(0)
        P = sum(diams_list) / len(diams_list)
        return int(best_L * 10000), int(P * 10000)

    res_al, res_aP = proc(read4(fa), 3)
    res_bl, res_bP = proc(read4(fb), 6)
    return res_al, res_aP, res_bl, res_bP

if __name__ == "__main__":
    cax, cay, cbx, cby = solve_task5("5_A.txt", "5_B.txt")
    print(f"{cax} {cay}")
    print(f"{cbx} {cby}")
    
    La, Pa, Lb, Pb = solve_task4("4_A.txt", "4_B.txt")
    print(f"{La} {Pa}")
    print(f"{Lb} {Pb}")

19991 40015
13609 44015
27100 49705
12200 68228
